In [1]:
import os
from pathlib import Path

CANDIDATE_DIR = Path('/workspace/outputs/candidates')

logics = ['timedecay', 'repurchase', 'variant', 'also_bought',
          'item2item_cf', 'bin_popular', 'purchase_interval']

cutoffs = ['2020-09-01', '2020-09-08', '2020-09-15', '2020-09-22']

print(f"{'ロジック':<20}", end='')
for c in cutoffs:
    print(f" {c}", end='')
print()
print("-" * 80)

for name in logics:
    print(f"{name:<20}", end='')
    for c in cutoffs:
        path = CANDIDATE_DIR / f"{name}_{c}.parquet"
        if path.exists():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"  ✅{size_mb:5.0f}MB", end='')
        else:
            print(f"       ❌ ", end='')
    print()

ロジック                 2020-09-01 2020-09-08 2020-09-15 2020-09-22
--------------------------------------------------------------------------------
timedecay             ✅  134MB  ✅  129MB  ✅  126MB  ✅  122MB
repurchase            ✅  209MB  ✅  211MB  ✅  212MB  ✅  214MB
variant               ✅  291MB  ✅  288MB  ✅  286MB  ✅  283MB
also_bought           ✅   17MB  ✅   16MB  ✅   17MB  ✅   16MB
item2item_cf          ✅   29MB  ✅   28MB  ✅   28MB  ✅   27MB
bin_popular           ✅   16MB  ✅   15MB  ✅   15MB  ✅   15MB
purchase_interval     ✅   24MB  ✅   24MB  ✅   24MB  ✅   24MB


In [2]:
import os
test_path = '/workspace/outputs/candidates/_test.txt'
try:
    with open(test_path, 'w') as f:
        f.write('test')
    os.remove(test_path)
    print("✅ 書き込み可能")
except Exception as e:
    print(f"❌ {e}")

✅ 書き込み可能


In [3]:
import polars as pl
import cudf
import pandas as pd
import numpy as np
import json
import time
import gc
import os
from pathlib import Path
from datetime import date, timedelta
from collections import defaultdict

DATA_DIR = Path('/workspace/data/raw')
CANDIDATE_DIR = Path('/workspace/outputs/candidates')
# CANDIDATE_DIR_RAP = Path('/home/rapids/candidates')
FEATURE_DIR = Path('/home/rapids/features')
MODEL_DIR = Path('/home/rapids/models')
CONVERTED_DIR = Path('/workspace/outputs/converted')
for d in [FEATURE_DIR, MODEL_DIR, CANDIDATE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EVAL_CUTOFFS = [12, 24, 48, 96, 120]
N_CANDIDATES = 100
TD_A, TD_B, TD_C, TD_D = 2.5e4, 1.5e5, 2e-1, 1e3

# 整数変換済みデータ読み込み
map_path = CONVERTED_DIR / 'id_maps.json'
with open(map_path, 'r') as f:
    maps = json.load(f)
customer_id_map = maps['customer_id_map']
customer_id_reverse = {int(v): k for k, v in customer_id_map.items()}
article_id_map = maps['article_id_map']
article_id_reverse = {int(v): k for k, v in article_id_map.items()}

transactions = pl.read_parquet(CONVERTED_DIR / 'transactions.parquet')
customers = pl.read_parquet(CONVERTED_DIR / 'customers.parquet')
articles = pl.read_parquet(CONVERTED_DIR / 'articles.parquet')

# cuDFに変換
transactions_gdf = cudf.from_pandas(transactions.to_pandas())
customers_gdf = cudf.from_pandas(customers.to_pandas())
articles_gdf = cudf.from_pandas(articles.to_pandas())

# best_weights読み込み
with open(CANDIDATE_DIR / 'best_weights_2020-09-15.json', 'r') as f:
    weights_data = json.load(f)
best_weights = weights_data['weights']
active_logics = {k: v for k, v in best_weights.items() if v > 0}

print(f"transactions: {len(transactions):,}行")
print(f"使用ロジック: {list(active_logics.keys())}")
print("セル1完了!")

transactions: 31,788,324行
使用ロジック: ['item2item_cf', 'bin_popular', 'purchase_interval', 'timedecay', 'repurchase', 'variant', 'also_bought']
セル1完了!


In [4]:
# ============================================================
# 共通関数
# ============================================================
def get_actuals(transactions, cutoff, label_days=7):
    label_end = cutoff + timedelta(days=label_days)
    grouped = (
        transactions
        .filter((pl.col('t_dat') > pl.lit(cutoff)) & (pl.col('t_dat') <= pl.lit(label_end)))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('actual'))
    )
    return {row['customer_id']: set(row['actual']) for row in grouped.iter_rows(named=True)}

def apk(actual_set, predicted, k):
    if not actual_set:
        return 0.0
    score = 0.0
    hits = 0
    seen = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in seen:
            continue
        seen.add(article_id)
        if article_id in actual_set:
            hits += 1
            score += hits / rank
    return score / min(len(actual_set), k)

def save_candidates(df, name, cutoff):
    path = CANDIDATE_DIR/ f"{name}_{cutoff}.parquet"
    df.write_parquet(path)
    print(f"  保存: {path}")
    return df

def load_candidates(name, cutoff):
    path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
    if path.exists():
        return pl.read_parquet(path)
    return None

def create_candidates_scored(cutoff, active_logics, valid_customers=None):
    scored_list = []
    for name, w in active_logics.items():
        df = load_candidates(name, cutoff)
        if df is None:
            continue
        if valid_customers is not None:
            df = df.filter(pl.col('customer_id').is_in(valid_customers))
        normalized = (
            df.select(['customer_id', 'article_id', 'score'])
            .join(
                df.group_by('customer_id').agg([
                    pl.col('score').max().alias('max_score'),
                    pl.col('score').min().alias('min_score'),
                ]),
                on='customer_id', how='left'
            )
            .with_columns(
                pl.when(pl.col('max_score') == pl.col('min_score'))
                .then(1.0)
                .otherwise(
                    (pl.col('score') - pl.col('min_score')) /
                    (pl.col('max_score') - pl.col('min_score'))
                )
                .alias('norm_score')
            )
            .with_columns((pl.col('norm_score') * w).alias('weighted_score'))
            .select(['customer_id', 'article_id', 'weighted_score'])
        )
        scored_list.append(normalized)
    combined = pl.concat(scored_list)
    return (
        combined
        .group_by(['customer_id', 'article_id'])
        .agg(pl.col('weighted_score').sum().alias('candidate_score'))
        .with_columns(
            pl.col('candidate_score').rank(method='ordinal', descending=True)
            .over('customer_id').alias('rank')
        )
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )

# ============================================================
# 候補生成関数
# ============================================================
def build_timedecay(transactions, cutoff, top_n=120):
    name = 'timedecay'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    df_train = df_train.with_columns(
        (pl.col('t_dat') +
         ((pl.lit(cutoff) - pl.col('t_dat')).dt.total_days() % 7
         ).cast(pl.Int32) * pl.duration(days=1)
        ).alias('ldbw')
    )
    weekly_sales = df_train.group_by(['ldbw', 'article_id']).agg(pl.len().alias('count'))
    df_train = df_train.join(weekly_sales, on=['ldbw', 'article_id'], how='left')
    last_week_sales = (
        weekly_sales.filter(pl.col('ldbw') == cutoff)
        .select(['article_id', 'count']).rename({'count': 'count_targ'})
    )
    df_train = (
        df_train
        .join(last_week_sales, on='article_id', how='left')
        .with_columns(pl.col('count_targ').fill_null(0))
        .with_columns((pl.col('count_targ') / pl.col('count')).alias('quotient'))
        .with_columns(
            (pl.lit(cutoff) - pl.col('t_dat')).dt.total_days().clip(lower_bound=1).alias('days_ago')
        )
        .with_columns(
            (
                pl.col('quotient') * (
                    TD_A / pl.col('days_ago').cast(pl.Float64).sqrt() +
                    TD_B * (-TD_C * pl.col('days_ago').cast(pl.Float64)).exp() - TD_D
                ).clip(lower_bound=0)
            ).alias('score')
        )
    )
    result = (
        df_train.group_by(['customer_id', 'article_id'])
        .agg(pl.col('score').sum())
        .filter(pl.col('score') > 0)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_repurchase(transactions, cutoff, top_n=120):
    name = 'repurchase'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    result = (
        df_train
        .group_by(['customer_id', 'article_id'])
        .agg(pl.max('t_dat').alias('last_date'))
        .with_columns(
            pl.col('last_date').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .with_columns(
            (1.0 / (1.0 + (pl.lit(cutoff) - pl.col('last_date')).dt.total_days().cast(pl.Float64))).alias('score')
        )
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_variant(transactions, cutoff, timedecay_df=None, top_n=30):
    name = 'variant'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    if timedecay_df is None:
        timedecay_df = load_candidates('timedecay', cutoff)
    articles_product = articles.select(['article_id', 'product_code'])
    result = (
        timedecay_df
        .filter(pl.col('rank') <= 50)
        .select(['customer_id', 'article_id', 'score'])
        .join(articles_product, on='article_id', how='left')
        .join(
            articles_product.rename({'article_id': 'variant_article_id'}),
            on='product_code', how='left'
        )
        .filter(pl.col('article_id') != pl.col('variant_article_id'))
        .drop_nulls(subset=['variant_article_id'])
        .group_by(['customer_id', pl.col('variant_article_id').alias('article_id')])
        .agg(pl.col('score').sum().alias('score'))
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_also_bought(transactions, cutoff, top_n=20, history_days=180):
    name = 'also_bought'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中（直近{history_days}日）...")
    start_date = cutoff - timedelta(days=history_days)
    df_train = transactions.filter(
        (pl.col('t_dat') > pl.lit(start_date)) & (pl.col('t_dat') <= pl.lit(cutoff))
    )
    months = (
        df_train.select(pl.col('t_dat').dt.truncate('1mo').alias('month'))
        .unique().sort('month').to_series().to_list()
    )
    print(f"    対象月数: {len(months)}")
    all_pairs = []
    for i, month in enumerate(months):
        month_end = month + timedelta(days=31)
        df_month = df_train.filter(
            (pl.col('t_dat') >= pl.lit(month)) & (pl.col('t_dat') < pl.lit(month_end))
        )
        if len(df_month) == 0:
            continue
        pairs = (
            df_month.select(['customer_id', 't_dat', 'article_id'])
            .join(df_month.select(['customer_id', 't_dat', 'article_id']), on=['customer_id', 't_dat'], how='inner')
            .filter(pl.col('article_id') != pl.col('article_id_right'))
            .group_by(['article_id', 'article_id_right'])
            .agg(pl.len().alias('count'))
        )
        all_pairs.append(pairs)
        print(f"    進捗: {i+1}/{len(months)}ヶ月")
    combined = (
        pl.concat(all_pairs)
        .group_by(['article_id', 'article_id_right'])
        .agg(pl.col('count').sum())
        .filter(pl.col('count') >= 10)
        .sort(['article_id', 'count'], descending=[False, True])
    )
    also_bought_dict = defaultdict(list)
    for row in combined.iter_rows(named=True):
        also_bought_dict[row['article_id']].append((row['article_id_right'], row['count']))
    recent_start = cutoff - timedelta(days=30)
    user_history = (
        transactions.filter(
            (pl.col('t_dat') >= pl.lit(recent_start)) & (pl.col('t_dat') <= pl.lit(cutoff))
        )
        .group_by('customer_id')
        .agg(pl.col('article_id').alias('articles'))
    )
    rows_list = []
    for row in user_history.iter_rows(named=True):
        cid = row['customer_id']
        hist_arts = row['articles']
        purchased_set = set(hist_arts)
        candidates = {}
        for art in hist_arts:
            for paired_art, count in also_bought_dict.get(art, []):
                if paired_art not in purchased_set:
                    candidates[paired_art] = candidates.get(paired_art, 0) + count
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({'customer_id': cid, 'article_id': art, 'score': float(score), 'rank': rank})
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_item2item_cf(transactions, cutoff, top_n=20):
    name = 'item2item_cf'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    from scipy.sparse import csr_matrix
    from sklearn.preprocessing import normalize
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    user_item = df_train.group_by(['customer_id', 'article_id']).agg(pl.len().alias('count'))
    n_users = len(customer_id_map)
    n_items = len(article_id_map)
    row = user_item['customer_id'].to_numpy()
    col = user_item['article_id'].to_numpy()
    data = user_item['count'].to_numpy().astype(np.float32)
    uim = csr_matrix((data, (row, col)), shape=(n_users, n_items))
    iim_norm = normalize(uim.T.tocsr(), norm='l2')
    print(f"    行列サイズ: {iim_norm.shape}")
    start_date = cutoff - timedelta(days=30)
    recent_articles = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .select('article_id').unique().to_series().to_list()
    )
    print(f"    直近1ヶ月の購買商品数: {len(recent_articles):,}")
    print("    類似度辞書を事前計算中...")
    similar_items_dict = {}
    for i, art in enumerate(recent_articles):
        if i % 10000 == 0:
            print(f"      進捗: {i:,} / {len(recent_articles):,}")
        if art >= iim_norm.shape[0]:
            continue
        sim_scores = iim_norm[art].dot(iim_norm.T).toarray().flatten()
        top_items = np.argsort(sim_scores)[::-1][1:top_n+1]
        similar_items_dict[art] = [
            (int(idx), float(sim_scores[idx]))
            for idx in top_items if sim_scores[idx] > 0
        ]
    print(f"    類似度辞書完成: {len(similar_items_dict):,}商品")
    recent_hist = (
        df_train.filter(pl.col('t_dat') >= pl.lit(start_date))
        .group_by('customer_id')
        .agg(pl.col('article_id').unique().alias('articles'))
    )
    rows_list = []
    for i, row_data in enumerate(recent_hist.iter_rows(named=True)):
        if i % 100000 == 0:
            print(f"      進捗: {i:,} / {len(recent_hist):,}")
        cid = row_data['customer_id']
        hist_arts = row_data['articles']
        purchased_set = set(hist_arts)
        candidates = {}
        for art in hist_arts:
            for sim_art, sim_score in similar_items_dict.get(art, []):
                if sim_art not in purchased_set:
                    candidates[sim_art] = candidates.get(sim_art, 0) + sim_score
        sorted_cands = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
        for rank, (art, score) in enumerate(sorted_cands, start=1):
            rows_list.append({'customer_id': cid, 'article_id': art, 'score': score, 'rank': rank})
    result = pl.DataFrame(rows_list).with_columns([
        pl.col('customer_id').cast(pl.Int32),
        pl.col('article_id').cast(pl.Int32),
        pl.col('rank').cast(pl.Int32),
    ])
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_bin_popular(transactions, cutoff, top_n=50):
    name = 'bin_popular'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    recent = (
        df_train
        .filter(pl.col('t_dat') >= pl.lit(cutoff - timedelta(days=28)))
        .join(customers.select(['customer_id', 'age_bin']), on='customer_id', how='left')
    )
    bin_popular = (
        recent.group_by(['age_bin', 'article_id']).agg(pl.len().alias('count'))
        .with_columns(
            pl.col('count').rank(method='ordinal', descending=True).over('age_bin').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
    )
    result = (
        customers.select(['customer_id', 'age_bin'])
        .join(bin_popular.select(['age_bin', 'article_id', 'count', 'rank']), on='age_bin', how='left')
        .drop_nulls()
        .with_columns(pl.col('count').cast(pl.Float64).alias('score'))
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

def build_purchase_interval(transactions, cutoff, top_n=20):
    name = 'purchase_interval'
    cached = load_candidates(name, cutoff)
    if cached is not None:
        return cached
    print(f"  [{name}] 計算中...")
    df_train = transactions.filter(pl.col('t_dat') <= pl.lit(cutoff))
    purchase_dates = (
        df_train
        .sort(['customer_id', 'article_id', 't_dat'])
        .group_by(['customer_id', 'article_id'])
        .agg([
            pl.col('t_dat').sort().alias('dates'),
            pl.col('t_dat').max().alias('last_date'),
            pl.len().alias('purchase_count')
        ])
        .filter(pl.col('purchase_count') >= 2)
    )
    rows_list = []
    for row in purchase_dates.iter_rows(named=True):
        dates = row['dates']
        if len(dates) < 2:
            continue
        diffs = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]
        avg_interval = sum(diffs) / len(diffs)
        days_since_last = (cutoff - row['last_date']).days
        score = 1.0 / (1.0 + abs(days_since_last - avg_interval))
        rows_list.append({
            'customer_id': row['customer_id'],
            'article_id': row['article_id'],
            'score': score,
        })
    result = (
        pl.DataFrame(rows_list)
        .with_columns(
            pl.col('score').rank(method='ordinal', descending=True).over('customer_id').alias('rank')
        )
        .filter(pl.col('rank') <= top_n)
        .select(['customer_id', 'article_id', 'score', 'rank'])
        .with_columns([
            pl.col('customer_id').cast(pl.Int32),
            pl.col('article_id').cast(pl.Int32),
            pl.col('rank').cast(pl.Int32),
        ])
    )
    print(f"  [{name}] 候補数: {len(result):,}")
    return save_candidates(result, name, cutoff)

# ============================================================
# 特徴量作成関数（cuDF版・キャッシュ付き）
# ============================================================
def build_user_features(transactions_gdf, customers_gdf, cutoff):
    cache_path = FEATURE_DIR / f'user_features_{cutoff}.parquet'
    if cache_path.exists():
        print(f"  ユーザー特徴量: キャッシュ読み込み {cache_path}")
        return cudf.read_parquet(str(cache_path))
    print("  ユーザー特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    user_basic = df.groupby('customer_id').agg({
        'article_id': ['count', 'nunique'], 't_dat': ['max', 'min']
    })
    user_basic.columns = ['total_purchases', 'unique_articles', 'last_purchase_date', 'first_purchase_date']
    user_basic = user_basic.reset_index()
    user_basic['days_since_last_purchase'] = (cutoff_dt - user_basic['last_purchase_date']).dt.days
    user_basic['days_since_first_purchase'] = (cutoff_dt - user_basic['first_purchase_date']).dt.days
    user_basic = user_basic.drop(columns=['last_purchase_date', 'first_purchase_date'])
    for days in [7, 14, 30, 60]:
        start_dt = np.datetime64(cutoff - timedelta(days=days))
        recent = df[df['t_dat'] >= start_dt]
        recent_count = recent.groupby('customer_id').agg({'article_id': 'count'})
        recent_count.columns = [f'purchases_last_{days}d']
        recent_count = recent_count.reset_index()
        user_basic = user_basic.merge(recent_count, on='customer_id', how='left')
        user_basic[f'purchases_last_{days}d'] = user_basic[f'purchases_last_{days}d'].fillna(0)
    start_30 = np.datetime64(cutoff - timedelta(days=30))
    recent30 = df[df['t_dat'] >= start_30]
    recent30_unique = recent30.groupby('customer_id').agg({'article_id': 'nunique'})
    recent30_unique.columns = ['unique_articles_last_30d']
    recent30_unique = recent30_unique.reset_index()
    user_basic = user_basic.merge(recent30_unique, on='customer_id', how='left')
    user_basic['unique_articles_last_30d'] = user_basic['unique_articles_last_30d'].fillna(0)
    user_basic['purchase_frequency_per_week'] = (
        user_basic['total_purchases'] / (user_basic['days_since_first_purchase'] / 7.0).clip(lower=1)
    )
    basket = df.groupby(['customer_id', 't_dat']).agg({'article_id': 'count'})
    basket.columns = ['basket_size']
    basket = basket.reset_index()
    avg_basket = basket.groupby('customer_id').agg({'basket_size': 'mean'})
    avg_basket.columns = ['avg_basket_size']
    avg_basket = avg_basket.reset_index()
    user_basic = user_basic.merge(avg_basket, on='customer_id', how='left')
    user_basic['avg_basket_size'] = user_basic['avg_basket_size'].fillna(1.0)
    ua_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    ua_count.columns = ['purchase_count']
    ua_count = ua_count.reset_index()
    ua_count['is_repeat'] = (ua_count['purchase_count'] > 1).astype('int32')
    repeat_stats = ua_count.groupby('customer_id').agg({'is_repeat': 'mean'})
    repeat_stats.columns = ['repeat_rate']
    repeat_stats = repeat_stats.reset_index()
    user_basic = user_basic.merge(repeat_stats, on='customer_id', how='left')
    user_basic['repeat_rate'] = user_basic['repeat_rate'].fillna(0)
    hist_before = df[df['t_dat'] < start_30]
    hist_before_set = hist_before[['customer_id', 'article_id']].drop_duplicates()
    hist_before_set['is_old'] = 1
    recent30_all = df[df['t_dat'] >= start_30][['customer_id', 'article_id']].drop_duplicates()
    recent30_merged = recent30_all.merge(hist_before_set, on=['customer_id', 'article_id'], how='left')
    recent30_merged['is_old'] = recent30_merged['is_old'].fillna(0)
    recent30_merged['is_new'] = 1 - recent30_merged['is_old']
    new_rate = recent30_merged.groupby('customer_id').agg({'is_new': 'mean'})
    new_rate.columns = ['new_article_rate_30d']
    new_rate = new_rate.reset_index()
    user_basic = user_basic.merge(new_rate, on='customer_id', how='left')
    user_basic['new_article_rate_30d'] = user_basic['new_article_rate_30d'].fillna(0)
    articles_gdf_slim = cudf.read_parquet(str(CONVERTED_DIR / 'articles.parquet'), columns=['article_id', 'product_type_no'])
    df_with_pt = df.merge(articles_gdf_slim, on='article_id', how='left')
    cat_div = df_with_pt.groupby('customer_id').agg({'product_type_no': 'nunique'})
    cat_div.columns = ['category_diversity']
    cat_div = cat_div.reset_index()
    user_basic = user_basic.merge(cat_div, on='customer_id', how='left')
    user_basic['category_diversity'] = user_basic['category_diversity'].fillna(0)
    df_sorted = df[['customer_id', 't_dat']].sort_values(['customer_id', 't_dat'])
    df_sorted['prev_t_dat'] = df_sorted.groupby('customer_id')['t_dat'].shift(1)
    df_sorted = df_sorted.dropna(subset=['prev_t_dat'])
    df_sorted['interval_days'] = (df_sorted['t_dat'] - df_sorted['prev_t_dat']).dt.days
    avg_interval = df_sorted.groupby('customer_id').agg({'interval_days': 'mean'})
    avg_interval.columns = ['avg_purchase_interval']
    avg_interval = avg_interval.reset_index()
    user_basic = user_basic.merge(avg_interval, on='customer_id', how='left')
    user_basic['avg_purchase_interval'] = user_basic['avg_purchase_interval'].fillna(-1)
    user_features = user_basic.merge(
        customers_gdf[['customer_id', 'age', 'age_bin', 'FN', 'Active',
                        'club_member_status_enc', 'fashion_news_frequency_enc']],
        on='customer_id', how='left'
    )
    user_features.to_pandas().to_parquet(str(cache_path))
    elapsed = time.time() - start
    print(f"  ユーザー特徴量: {user_features.shape} （{elapsed/60:.1f}分）保存完了")
    return user_features

def build_article_features(transactions_gdf, articles_gdf, cutoff):
    cache_path = FEATURE_DIR / f'article_features_{cutoff}.parquet'
    if cache_path.exists():
        print(f"  商品特徴量: キャッシュ読み込み {cache_path}")
        return cudf.read_parquet(str(cache_path))
    print("  商品特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    article_basic = df.groupby('article_id').agg({
        'customer_id': ['count', 'nunique'], 't_dat': 'min'
    })
    article_basic.columns = ['total_sales', 'unique_buyers', 'first_sale_date']
    article_basic = article_basic.reset_index()
    article_basic['days_since_first_sale'] = (cutoff_dt - article_basic['first_sale_date']).dt.days
    article_basic = article_basic.drop(columns=['first_sale_date'])
    for days in [7, 14, 30, 60]:
        start_dt = np.datetime64(cutoff - timedelta(days=days))
        recent = df[df['t_dat'] >= start_dt]
        recent_count = recent.groupby('article_id').agg({'customer_id': 'count'})
        recent_count.columns = [f'sales_last_{days}d']
        recent_count = recent_count.reset_index()
        article_basic = article_basic.merge(recent_count, on='article_id', how='left')
        article_basic[f'sales_last_{days}d'] = article_basic[f'sales_last_{days}d'].fillna(0)
    article_basic['trend_7_14'] = article_basic['sales_last_7d'] / article_basic['sales_last_14d'].clip(lower=1)
    article_basic['trend_14_30'] = article_basic['sales_last_14d'] / article_basic['sales_last_30d'].clip(lower=1)
    article_basic['trend_30_60'] = article_basic['sales_last_30d'] / article_basic['sales_last_60d'].clip(lower=1)
    ua_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    ua_count.columns = ['purchase_count']
    ua_count = ua_count.reset_index()
    ua_count['is_repeat'] = (ua_count['purchase_count'] > 1).astype('int32')
    article_repeat = ua_count.groupby('article_id').agg({'purchase_count': 'mean', 'is_repeat': 'mean'})
    article_repeat.columns = ['avg_purchase_count_per_buyer', 'article_repeat_rate']
    article_repeat = article_repeat.reset_index()
    article_basic = article_basic.merge(article_repeat, on='article_id', how='left')
    article_basic['avg_purchase_count_per_buyer'] = article_basic['avg_purchase_count_per_buyer'].fillna(1.0)
    article_basic['article_repeat_rate'] = article_basic['article_repeat_rate'].fillna(0.0)
    art_attrs = articles_gdf[['article_id', 'product_type_no', 'garment_group_no',
                               'index_group_no', 'section_no', 'department_no', 'product_code']]
    article_basic = article_basic.merge(art_attrs, on='article_id', how='left')
    variant_count = articles_gdf.groupby('product_code').agg({'article_id': 'count'})
    variant_count.columns = ['variant_count']
    variant_count = variant_count.reset_index()
    article_basic = article_basic.merge(variant_count, on='product_code', how='left')
    article_basic['variant_count'] = article_basic['variant_count'].fillna(1)
    article_basic = article_basic.drop(columns=['product_code'])
    article_basic.to_pandas().to_parquet(str(cache_path))
    elapsed = time.time() - start
    print(f"  商品特徴量: {article_basic.shape} （{elapsed/60:.1f}分）保存完了")
    return article_basic

def build_user_article_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf):
    print("  ユーザー×商品特徴量作成中...")
    start = time.time()
    cutoff_dt = np.datetime64(cutoff)
    df = transactions_gdf[transactions_gdf['t_dat'] <= cutoff_dt]
    result = candidates_gdf.copy()
    source_names = ['timedecay', 'repurchase', 'variant', 'also_bought',
                    'item2item_cf', 'bin_popular', 'purchase_interval']
    for name in source_names:
        path = CANDIDATE_DIR / f"{name}_{cutoff}.parquet"
        if not path.exists():
            result[f'is_{name}'] = 0
            continue
        source_df = cudf.read_parquet(str(path), columns=['customer_id', 'article_id'])
        source_df = source_df.drop_duplicates()
        source_df[f'is_{name}'] = 1
        result = result.merge(source_df, on=['customer_id', 'article_id'], how='left')
        result[f'is_{name}'] = result[f'is_{name}'].fillna(0).astype('int32')
        print(f"    is_{name}: 完了")
    repeat_count = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'count'})
    repeat_count.columns = ['repeat_count']
    repeat_count = repeat_count.reset_index()
    result = result.merge(repeat_count, on=['customer_id', 'article_id'], how='left')
    result['repeat_count'] = result['repeat_count'].fillna(0)
    last_purchase_article = df.groupby(['customer_id', 'article_id']).agg({'t_dat': 'max'})
    last_purchase_article.columns = ['last_purchase_date']
    last_purchase_article = last_purchase_article.reset_index()
    last_purchase_article['days_since_last_purchase_article'] = (
        cutoff_dt - last_purchase_article['last_purchase_date']
    ).dt.days
    last_purchase_article = last_purchase_article.drop(columns=['last_purchase_date'])
    result = result.merge(last_purchase_article, on=['customer_id', 'article_id'], how='left')
    result['days_since_last_purchase_article'] = result['days_since_last_purchase_article'].fillna(-1)
    df_age = df.merge(customers_gdf[['customer_id', 'age_bin']], on='customer_id', how='left')
    age_bin_counts = df_age.groupby(['age_bin', 'article_id']).agg({'customer_id': 'count'})
    age_bin_counts.columns = ['age_bin_article_count']
    age_bin_counts = age_bin_counts.reset_index()
    age_bin_totals = df_age.groupby('age_bin').agg({'customer_id': 'count'})
    age_bin_totals.columns = ['age_bin_total']
    age_bin_totals = age_bin_totals.reset_index()
    age_bin_rate = age_bin_counts.merge(age_bin_totals, on='age_bin', how='left')
    age_bin_rate['age_bin_purchase_rate'] = age_bin_rate['age_bin_article_count'] / age_bin_rate['age_bin_total']
    age_bin_rate = age_bin_rate[['age_bin', 'article_id', 'age_bin_purchase_rate']]
    result = result.merge(customers_gdf[['customer_id', 'age_bin']], on='customer_id', how='left')
    result = result.merge(age_bin_rate, on=['age_bin', 'article_id'], how='left')
    result['age_bin_purchase_rate'] = result['age_bin_purchase_rate'].fillna(0)
    result = result.drop(columns=['age_bin'])
    art_cats = articles_gdf[['article_id', 'product_type_no', 'garment_group_no', 'index_group_no']]
    df_cat = df.merge(art_cats, on='article_id', how='left')
    user_total = df.groupby('customer_id').agg({'article_id': 'count'})
    user_total.columns = ['user_total']
    user_total = user_total.reset_index()
    for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
        user_cat = df_cat.groupby(['customer_id', cat_col]).agg({'article_id': 'count'})
        user_cat.columns = [f'user_{cat_col}_count']
        user_cat = user_cat.reset_index()
        user_cat = user_cat.merge(user_total, on='customer_id', how='left')
        user_cat[f'user_{cat_col}_ratio'] = user_cat[f'user_{cat_col}_count'] / user_cat['user_total']
        user_cat = user_cat.drop(columns=['user_total'])
        art_cat_col = articles_gdf[['article_id', cat_col]]
        result = result.merge(art_cat_col, on='article_id', how='left')
        result = result.merge(user_cat, on=['customer_id', cat_col], how='left')
        result[f'user_{cat_col}_count'] = result[f'user_{cat_col}_count'].fillna(0)
        result[f'user_{cat_col}_ratio'] = result[f'user_{cat_col}_ratio'].fillna(0)
        result = result.drop(columns=[cat_col])
        print(f"    {cat_col}: 完了")
    for cat_col in ['product_type_no', 'garment_group_no']:
        last_cat = df_cat.groupby(['customer_id', cat_col]).agg({'t_dat': 'max'})
        last_cat.columns = ['last_cat_date']
        last_cat = last_cat.reset_index()
        last_cat[f'days_since_last_{cat_col}'] = (cutoff_dt - last_cat['last_cat_date']).dt.days
        last_cat = last_cat.drop(columns=['last_cat_date'])
        art_cat_col = articles_gdf[['article_id', cat_col]]
        result = result.merge(art_cat_col, on='article_id', how='left')
        result = result.merge(last_cat, on=['customer_id', cat_col], how='left')
        result[f'days_since_last_{cat_col}'] = result[f'days_since_last_{cat_col}'].fillna(-1)
        result = result.drop(columns=[cat_col])
        print(f"    days_since_last_{cat_col}: 完了")
    elapsed = time.time() - start
    print(f"  ユーザー×商品特徴量: {result.shape} （{elapsed/60:.1f}分）")
    return result

def build_all_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf):
    print(f"\n特徴量作成開始（cutoff={cutoff}）...")
    user_feat = build_user_features(transactions_gdf, customers_gdf, cutoff)
    article_feat = build_article_features(transactions_gdf, articles_gdf, cutoff)
    ua_feat = build_user_article_features(transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf)
    result = ua_feat
    result = result.merge(user_feat, on='customer_id', how='left')
    result = result.merge(article_feat, on='article_id', how='left')
    for col in result.columns:
        if result[col].dtype in ['float32', 'float64']:
            result[col] = result[col].fillna(0.0)
        elif result[col].dtype in ['int32', 'int64']:
            result[col] = result[col].fillna(-1)
    print(f"\n特徴量作成完了: {result.shape}")
    return result

print("セル2完了!")

セル2完了!


In [13]:
# ============================================================
# 古い09-08候補ファイルを削除
# ============================================================
logics = ['timedecay', 'repurchase', 'variant', 'also_bought',
          'item2item_cf', 'bin_popular', 'purchase_interval']

print("09-08の古い候補ファイルを削除中...")
for name in logics:
    path = CANDIDATE_DIR / f"{name}_2020-09-08.parquet"
    if path.exists():
        os.remove(str(path))
        print(f"  削除: {path}")

# 09-08の古い特徴量キャッシュも削除
for prefix in ['user_features', 'article_features']:
    path = FEATURE_DIR / f"{prefix}_2020-09-08.parquet"
    if path.exists():
        os.remove(str(path))
        print(f"  削除: {path}")

# ============================================================
# 候補生成（item2item_cfは最後）
# ============================================================
cutoffs_to_generate = [date(2020, 8, 25), date(2020, 9, 1), date(2020, 9, 8)]

for cutoff in cutoffs_to_generate:
    print(f"\n{'='*50}")
    print(f"候補生成: cutoff={cutoff}（item2item_cf以外）")
    print(f"{'='*50}")
    
    print("\n① timedecay...")
    td_df = build_timedecay(transactions, cutoff)
    
    print("\n② repurchase...")
    rp_df = build_repurchase(transactions, cutoff)
    
    print("\n③ variant...")
    var_df = build_variant(transactions, cutoff, td_df)
    
    print("\n④ also_bought...")
    ab_df = build_also_bought(transactions, cutoff)
    
    print("\n⑤ bin_popular...")
    bin_df = build_bin_popular(transactions, cutoff)
    
    print("\n⑥ purchase_interval...")
    pi_df = build_purchase_interval(transactions, cutoff)
    
    print(f"\ncutoff={cutoff}（item2item_cf以外）完了!")

# item2item_cfを最後にまとめて実行
print(f"\n{'='*50}")
print("item2item_cf を全cutoffで生成")
print(f"{'='*50}")
for cutoff in cutoffs_to_generate:
    print(f"\n  cutoff={cutoff}...")
    cf_df = build_item2item_cf(transactions, cutoff)

print("\n全候補生成完了!")

# 確認
print("\n【候補ファイル一覧】")
for cutoff in [date(2020, 8, 25), date(2020, 9, 1), date(2020, 9, 8),
               date(2020, 9, 15), date(2020, 9, 22)]:
    print(f"\ncutoff={cutoff}:")
    for name in logics:
        path = CANDIDATE_DIR / f"{name}_{cutoff}_v5.parquet"
        if path.exists():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"  ✅ {name:<20} {size_mb:>6.1f}MB")
        else:
            print(f"  ❌ {name}")

09-08の古い候補ファイルを削除中...
  削除: /workspace/outputs/candidates/timedecay_2020-09-08.parquet
  削除: /workspace/outputs/candidates/repurchase_2020-09-08.parquet
  削除: /workspace/outputs/candidates/variant_2020-09-08.parquet
  削除: /workspace/outputs/candidates/also_bought_2020-09-08.parquet
  削除: /workspace/outputs/candidates/item2item_cf_2020-09-08.parquet
  削除: /workspace/outputs/candidates/bin_popular_2020-09-08.parquet
  削除: /workspace/outputs/candidates/purchase_interval_2020-09-08.parquet
  削除: /home/rapids/features/user_features_2020-09-08.parquet
  削除: /home/rapids/features/article_features_2020-09-08.parquet

候補生成: cutoff=2020-08-25（item2item_cf以外）

① timedecay...
  [timedecay] 計算中...
  [timedecay] 候補数: 11,278,794
  保存: /workspace/outputs/candidates/timedecay_2020-08-25.parquet

② repurchase...
  [repurchase] 計算中...
  [repurchase] 候補数: 25,056,512
  保存: /workspace/outputs/candidates/repurchase_2020-08-25.parquet

③ variant...
  [variant] 計算中...
  [variant] 候補数: 23,831,853
  保存: /workspa

In [14]:
import subprocess
result = subprocess.run(['ls', '-la', '/workspace/outputs/candidates/'], capture_output=True, text=True)
print(result.stdout)

total 5051364
drwxrwxrwx 1 root   root        512 Jun  9 11:36 .
drwxrwxrwx 1 root   root        512 Jun  2 06:46 ..
-rw-r--r-- 1 rapids conda  17564350 Jun  9 00:55 also_bought_2020-08-25.parquet
-rw-r--r-- 1 rapids conda  17385436 Jun  9 00:56 also_bought_2020-09-01.parquet
-rw-r--r-- 1 rapids conda  16688676 Jun  9 00:57 also_bought_2020-09-08.parquet
-rwxrwxrwx 1 root   root   17493099 May 29 13:18 also_bought_2020-09-15.parquet
-rwxrwxrwx 1 root   root   16398900 Jun  1 07:22 also_bought_2020-09-22.parquet
-rwxrwxrwx 1 root   root        238 Jun  1 06:17 best_weights.json
-rwxrwxrwx 1 root   root        313 Jun  1 07:06 best_weights_2020-09-15.json
-rw-r--r-- 1 rapids conda  15655401 Jun  9 00:55 bin_popular_2020-08-25.parquet
-rw-r--r-- 1 rapids conda  16378507 Jun  9 00:56 bin_popular_2020-09-01.parquet
-rw-r--r-- 1 rapids conda  15667293 Jun  9 00:57 bin_popular_2020-09-08.parquet
-rwxrwxrwx 1 root   root   15988910 May 29 14:17 bin_popular_2020-09-15.parquet
-rwxrwxrwx 1 root 

In [15]:
print(f"CANDIDATE_DIR: {CANDIDATE_DIR}")
print(f"exists: {CANDIDATE_DIR.exists()}")

# テスト
test = CANDIDATE_DIR / 'timedecay_2020-09-08.parquet'
print(f"test path: {test}")
print(f"test exists: {test.exists()}")

# 直接読み込みテスト
df = pl.read_parquet(str(test))
print(f"読み込み成功: {len(df):,}行")

CANDIDATE_DIR: /workspace/outputs/candidates
exists: True
test path: /workspace/outputs/candidates/timedecay_2020-09-08.parquet
test exists: True
読み込み成功: 10,613,527行


In [5]:
import os
os.chdir('/home/rapids')
print(f"作業ディレクトリ: {os.getcwd()}")

作業ディレクトリ: /home/rapids


In [4]:
# ============================================================
# 開発：3fold学習（ディスク保存方式）
# ============================================================
from catboost import CatBoost, Pool
import os

os.chdir('/home/rapids')

DEV_TRAIN_CUTOFFS = [date(2020, 8, 25), date(2020, 9, 1), date(2020, 9, 8)]
DEV_VALID_CUTOFF = date(2020, 9, 15)

# STEP1: 各foldの特徴量を個別に保存
for cutoff in DEV_TRAIN_CUTOFFS:
    fold_path = FEATURE_DIR / f'train_fold_{cutoff}.parquet'
    if fold_path.exists():
        print(f"cutoff={cutoff}: キャッシュあり ✅")
        continue
    
    print(f"\n{'='*50}")
    print(f"cutoff={cutoff}")
    print(f"{'='*50}")
    
    actuals = get_actuals(transactions, cutoff)
    valid_customers = list(actuals.keys())
    print(f"  customers: {len(valid_customers):,}")
    
    scored = (
        create_candidates_scored(cutoff, active_logics, valid_customers)
        .filter(pl.col('rank') <= N_CANDIDATES)
    )
    
    actuals_df = pl.DataFrame([
        {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
        for cid, aids in actuals.items()
        for aid in aids
    ]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])
    
    scored = scored.join(actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('label').fill_null(0)
    )
    print(f"  候補数: {len(scored):,} / 正例数: {scored['label'].sum():,}")
    
    candidates_gdf = cudf.from_pandas(scored.to_pandas())
    features_gdf = build_all_features(
        transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf
    )
    
    # ディスクに保存
    features_gdf.to_pandas().to_parquet(str(fold_path))
    print(f"  保存完了: {fold_path}")
    
    del candidates_gdf, features_gdf, scored; gc.collect()

print("\n全fold保存完了!")


cutoff=2020-08-25
  customers: 80,253
  候補数: 7,331,572 / 正例数: 22,230

特徴量作成開始（cutoff=2020-08-25）...
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-08-25.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-08-25.parquet
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了
    is_bin_popular: 完了
    is_purchase_interval: 完了
    product_type_no: 完了
    garment_group_no: 完了
    index_group_no: 完了
    days_since_last_product_type_no: 完了
    days_since_last_garment_group_no: 完了
  ユーザー×商品特徴量: (7331572, 23) （0.3分）

特徴量作成完了: (7331572, 62)
  保存完了: /home/rapids/features/train_fold_2020-08-25.parquet

cutoff=2020-09-01
  customers: 75,822
  候補数: 6,914,504 / 正例数: 24,158

特徴量作成開始（cutoff=2020-09-01）...
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-09-01.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-09-01.parquet
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了


In [8]:
# STEP2: 保存済みfoldを読み込んでCatBoost学習
print("保存済みfoldを読み込み中...")
DEV_TRAIN_CUTOFFS = [date(2020, 8, 25), date(2020, 9, 1), date(2020, 9, 8)]
DEV_VALID_CUTOFF = date(2020, 9, 15)
from catboost import CatBoost, Pool
train_dfs = []
for cutoff in DEV_TRAIN_CUTOFFS:
    fold_path = FEATURE_DIR / f'train_fold_{cutoff}.parquet'
    df = pd.read_parquet(str(fold_path))
    print(f"  {cutoff}: {len(df):,}行")
    train_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True)
del train_dfs; gc.collect()

exclude_cols = ['customer_id', 'article_id', 'label']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

with open(str(MODEL_DIR / 'feature_cols.json'), 'w') as f:
    json.dump(feature_cols, f)

print(f"\n学習データ合計: {len(train_df):,}行")
print(f"正例数: {train_df['label'].sum():,}")
print(f"特徴量数: {len(feature_cols)}")

print("\nCatBoost学習中...")
train_df_sorted = train_df.sort_values('customer_id')
del train_df; gc.collect()

train_pool = Pool(
    data=train_df_sorted[feature_cols],
    label=train_df_sorted['label'],
    group_id=train_df_sorted['customer_id']
)

model_dev = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 50,
    'early_stopping_rounds': 30,
})
model_dev.fit(train_pool)
print("学習完了!")

model_dev.save_model(str(MODEL_DIR / 'catboost_model_dev.cbm'))
print(f"モデル保存: {MODEL_DIR / 'catboost_model_dev.cbm'}")

importance = model_dev.get_feature_importance(train_pool, type='PredictionValuesChange')
feat_imp = sorted(zip(feature_cols, importance), key=lambda x: -x[1])
print("\n【特徴量重要度 TOP20】")
for i, (name, imp) in enumerate(feat_imp[:20], 1):
    print(f"  {i:>2}. {name:<35} {imp:.4f}")

del train_df_sorted, train_pool; gc.collect()
print("\n学習完了!")

保存済みfoldを読み込み中...
  2020-08-25: 7,331,572行
  2020-09-01: 6,914,504行
  2020-09-08: 6,574,658行

学習データ合計: 20,820,734行
正例数: 70,463
特徴量数: 59

CatBoost学習中...
0:	total: 7.56s	remaining: 37m 40s
50:	total: 2m 28s	remaining: 12m 7s
100:	total: 4m 45s	remaining: 9m 22s
150:	total: 7m 1s	remaining: 6m 56s
200:	total: 9m 18s	remaining: 4m 35s
250:	total: 11m 35s	remaining: 2m 15s
299:	total: 13m 50s	remaining: 0us
学習完了!
モデル保存: /home/rapids/models/catboost_model_dev.cbm

【特徴量重要度 TOP20】
   1. sales_last_7d                       14.7199
   2. sales_last_14d                      10.5011
   3. candidate_score                     7.3538
   4. trend_7_14                          6.4078
   5. trend_14_30                         6.0674
   6. sales_last_30d                      5.5451
   7. days_since_first_sale               3.5810
   8. user_index_group_no_ratio           3.1106
   9. rank                                2.7585
  10. trend_30_60                         2.5596
  11. user_garment_group_no_ra

In [9]:
# STEP3: valid評価
print(f"\nvalid評価中（cutoff={DEV_VALID_CUTOFF}）...")

try:
    model_dev
except NameError:
    model_dev = CatBoost()
    model_dev.load_model(str(MODEL_DIR / 'catboost_model_dev.cbm'))
    with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
        feature_cols = json.load(f)
    print("  モデルをロードしました")

valid_actuals = get_actuals(transactions, DEV_VALID_CUTOFF)
valid_customers = list(valid_actuals.keys())
print(f"  valid_customers: {len(valid_customers):,}")

valid_scored = (
    create_candidates_scored(DEV_VALID_CUTOFF, active_logics, valid_customers)
    .filter(pl.col('rank') <= N_CANDIDATES)
)
valid_actuals_df = pl.DataFrame([
    {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
    for cid, aids in valid_actuals.items()
    for aid in aids
]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])

valid_scored = valid_scored.join(valid_actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
    pl.col('label').fill_null(0)
)
print(f"  候補数: {len(valid_scored):,} / 正例数: {valid_scored['label'].sum():,}")

valid_candidates_gdf = cudf.from_pandas(valid_scored.to_pandas())
valid_features_gdf = build_all_features(
    transactions_gdf, articles_gdf, customers_gdf, DEV_VALID_CUTOFF, valid_candidates_gdf
)

valid_df = valid_features_gdf.to_pandas()
del valid_candidates_gdf, valid_features_gdf; gc.collect()

valid_df['pred_score'] = model_dev.predict(valid_df[feature_cols])

metric_acc = {k: {'recall': [], 'map': []} for k in EVAL_CUTOFFS}
for cid, group in valid_df.groupby('customer_id'):
    actual_set = valid_actuals.get(cid, set())
    if not actual_set:
        continue
    prediction = group.sort_values('pred_score', ascending=False)['article_id'].tolist()
    for k in EVAL_CUTOFFS:
        hits = len(set(prediction[:k]) & actual_set)
        metric_acc[k]['recall'].append(hits / len(actual_set))
        metric_acc[k]['map'].append(apk(actual_set, prediction, k))

print("\n" + "="*50)
print("【valid評価】")
print("="*50)
print(f"{'k':>6} {'recall':>10} {'MAP':>10}")
print("-"*30)
for k in EVAL_CUTOFFS:
    recall = np.mean(metric_acc[k]['recall'])
    map_k = np.mean(metric_acc[k]['map'])
    print(f"{k:>6} {recall:>10.6f} {map_k:>10.6f}")

print(f"\n【前回（1fold）との比較】")
print(f"  1fold  MAP@12: 0.034213")
print(f"  3fold  MAP@12: {np.mean(metric_acc[12]['map']):.6f}")

del valid_df; gc.collect()
print("\nvalid評価完了!")


valid評価中（cutoff=2020-09-15）...
  valid_customers: 68,984
  候補数: 6,274,713 / 正例数: 23,513

特徴量作成開始（cutoff=2020-09-15）...
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-09-15.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-09-15.parquet
  ユーザー×商品特徴量作成中...
    is_timedecay: 完了
    is_repurchase: 完了
    is_variant: 完了
    is_also_bought: 完了
    is_item2item_cf: 完了
    is_bin_popular: 完了
    is_purchase_interval: 完了
    product_type_no: 完了
    garment_group_no: 完了
    index_group_no: 完了
    days_since_last_product_type_no: 完了
    days_since_last_garment_group_no: 完了
  ユーザー×商品特徴量: (6274713, 23) （0.3分）

特徴量作成完了: (6274713, 62)

【valid評価】
     k     recall        MAP
------------------------------
    12   0.072915   0.034905
    24   0.095269   0.036400
    48   0.121299   0.037320
    96   0.133072   0.037570
   120   0.133098   0.037571

【前回（1fold）との比較】
  1fold  MAP@12: 0.034213
  3fold  MAP@12: 0.034905

valid評価完了!


In [10]:
# ============================================================
# 提出用再学習（09-01, 09-08, 09-15）ディスク保存方式
# ============================================================
SUB_TRAIN_CUTOFFS = [date(2020, 9, 1), date(2020, 9, 8), date(2020, 9, 15)]

# STEP1: 各foldの特徴量を個別に保存
for cutoff in SUB_TRAIN_CUTOFFS:
    fold_path = FEATURE_DIR / f'sub_fold_{cutoff}.parquet'
    if fold_path.exists():
        print(f"cutoff={cutoff}: キャッシュあり ✅")
        continue
    
    print(f"\n{'='*50}")
    print(f"cutoff={cutoff}")
    print(f"{'='*50}")
    
    actuals = get_actuals(transactions, cutoff)
    valid_customers = list(actuals.keys())
    print(f"  customers: {len(valid_customers):,}")
    
    scored = (
        create_candidates_scored(cutoff, active_logics, valid_customers)
        .filter(pl.col('rank') <= N_CANDIDATES)
    )
    
    actuals_df = pl.DataFrame([
        {'customer_id': int(cid), 'article_id': int(aid), 'label': 1}
        for cid, aids in actuals.items()
        for aid in aids
    ]).with_columns([pl.col('customer_id').cast(pl.Int32), pl.col('article_id').cast(pl.Int32)])
    
    scored = scored.join(actuals_df, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('label').fill_null(0)
    )
    print(f"  候補数: {len(scored):,} / 正例数: {scored['label'].sum():,}")
    
    candidates_gdf = cudf.from_pandas(scored.to_pandas())
    features_gdf = build_all_features(
        transactions_gdf, articles_gdf, customers_gdf, cutoff, candidates_gdf
    )
    
    features_gdf.to_pandas().to_parquet(str(fold_path))
    print(f"  保存完了: {fold_path}")
    
    del candidates_gdf, features_gdf, scored; gc.collect()

print("\n全fold保存完了!")

cutoff=2020-09-01: キャッシュあり ✅
cutoff=2020-09-08: キャッシュあり ✅
cutoff=2020-09-15: キャッシュあり ✅

全fold保存完了!


In [8]:
# STEP2: 保存済みfoldを読み込んでCatBoost学習
SUB_TRAIN_CUTOFFS = [date(2020, 9, 1), date(2020, 9, 8), date(2020, 9, 15)]
from catboost import CatBoost, Pool
import os

os.chdir('/home/rapids')
# STEP2: 保存済みfoldを読み込んでCatBoost学習
# STEP2: 保存済みfoldを読み込んでCatBoost学習
print("保存済みfoldを読み込み中...")
train_dfs = []
for cutoff in SUB_TRAIN_CUTOFFS:
    fold_path = FEATURE_DIR / f'sub_fold_{cutoff}.parquet'
    df = pd.read_parquet(str(fold_path))
    print(f"  {cutoff}: {len(df):,}行")
    train_dfs.append(df)

train_df = pd.concat(train_dfs, ignore_index=True)
del train_dfs; gc.collect()

with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)

print(f"\n学習データ合計: {len(train_df):,}行")
print(f"正例数: {train_df['label'].sum():,}")
print(f"特徴量数: {len(feature_cols)}")

print("\nCatBoost学習中...")
train_df_sorted = train_df.sort_values('customer_id')
del train_df; gc.collect()

train_pool = Pool(
    data=train_df_sorted[feature_cols],
    label=train_df_sorted['label'],
    group_id=train_df_sorted['customer_id']
)

model_sub = CatBoost({
    'loss_function': 'YetiRank',
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'verbose': 50,
    'early_stopping_rounds': 30,
})
model_sub.fit(train_pool)
print("学習完了!")

model_sub.save_model(str(MODEL_DIR / 'catboost_model_v4.cbm'))
print(f"モデル保存: {MODEL_DIR / 'catboost_model_v4.cbm'}")

del train_df_sorted, train_pool; gc.collect()
print("\n提出用モデル学習完了!")

保存済みfoldを読み込み中...
  2020-09-01: 6,914,504行
  2020-09-08: 6,574,658行
  2020-09-15: 6,274,713行

学習データ合計: 19,763,875行
正例数: 71,751
特徴量数: 57

CatBoost学習中...
0:	total: 13.4s	remaining: 1h 6m 57s
50:	total: 2m 33s	remaining: 12m 28s
100:	total: 4m 51s	remaining: 9m 34s
150:	total: 7m 8s	remaining: 7m 2s
200:	total: 9m 26s	remaining: 4m 39s
250:	total: 11m 44s	remaining: 2m 17s
299:	total: 14m	remaining: 0us
学習完了!
モデル保存: /home/rapids/models/catboost_model_v4.cbm

提出用モデル学習完了!


In [ ]:
# # STEP2: 保存済みfoldを読み込んでCatBoost学習（メモリ軽量版）
# print("保存済みfoldを読み込み中...")

# with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
#     feature_cols = json.load(f)

# train_dfs = []
# for cutoff in SUB_TRAIN_CUTOFFS:
#     fold_path = FEATURE_DIR / f'sub_fold_{cutoff}.parquet'
#     df = pd.read_parquet(str(fold_path))
#     df = df[df['rank'] <= 50]
#     print(f"  {cutoff}: {len(df):,}行")
#     train_dfs.append(df)

# train_df = pd.concat(train_dfs, ignore_index=True)
# del train_dfs; gc.collect()

# print(f"\n学習データ合計: {len(train_df):,}行")
# print(f"正例数: {train_df['label'].sum():,}")

# print("\nCatBoost学習中...")
# train_df_sorted = train_df.sort_values('customer_id')
# del train_df; gc.collect()

# train_pool = Pool(
#     data=train_df_sorted[feature_cols],
#     label=train_df_sorted['label'],
#     group_id=train_df_sorted['customer_id']
# )
# del train_df_sorted; gc.collect()

# model_sub = CatBoost({
#     'loss_function': 'YetiRank',
#     'iterations': 300,
#     'learning_rate': 0.05,
#     'depth': 6,
#     'verbose': 50,
#     'early_stopping_rounds': 30,
# })
# model_sub.fit(train_pool)
# print("学習完了!")

# model_sub.save_model(str(MODEL_DIR / 'catboost_model_v3.cbm'))
# print(f"モデル保存: {MODEL_DIR / 'catboost_model_v3.cbm'}")

# del train_pool; gc.collect()
# print("\n提出用モデル学習完了!")

保存済みfoldを読み込み中...
  2020-09-01: 3,791,100行
  2020-09-08: 3,600,950行
  2020-09-15: 3,449,189行

学習データ合計: 10,841,239行
正例数: 53,800

CatBoost学習中...
0:	total: 1.35s	remaining: 6m 44s
50:	total: 1m 5s	remaining: 5m 21s
100:	total: 2m 10s	remaining: 4m 16s
150:	total: 3m 14s	remaining: 3m 12s
200:	total: 4m 18s	remaining: 2m 7s
250:	total: 5m 22s	remaining: 1m 3s
299:	total: 6m 25s	remaining: 0us
学習完了!
モデル保存: /home/rapids/models/catboost_model_v3.cbm

提出用モデル学習完了!


In [9]:
# STEP3: 提出ファイル作成
print("\n提出ファイル作成中...")

from catboost import CatBoost

try:
    model_sub
except NameError:
    model_sub = CatBoost()
    model_sub.load_model(str(MODEL_DIR / 'catboost_model_v4.cbm'))
    with open(str(MODEL_DIR / 'feature_cols.json'), 'r') as f:
        feature_cols = json.load(f)
    print("  モデルをロードしました")

SUB_CUTOFF = date(2020, 9, 22)
print(f"  提出用cutoff: {SUB_CUTOFF}")

scored_df = pl.read_parquet(str(CANDIDATE_DIR / f'candidates_scored_{SUB_CUTOFF}.parquet'))
print(f"  全候補数: {len(scored_df):,}行")

user_feat = build_user_features(transactions_gdf, customers_gdf, SUB_CUTOFF)
article_feat = build_article_features(transactions_gdf, articles_gdf, SUB_CUTOFF)
user_feat_pd = user_feat.to_pandas()
article_feat_pd = article_feat.to_pandas()
del user_feat, article_feat; gc.collect()

# 事前計算
print("  事前計算中...")
df_train_pl = transactions.filter(pl.col('t_dat') <= pl.lit(SUB_CUTOFF))

repeat_count_pl = df_train_pl.group_by(['customer_id', 'article_id']).agg(pl.len().alias('repeat_count'))

last_purchase_article_pl = (
    df_train_pl.group_by(['customer_id', 'article_id'])
    .agg(pl.col('t_dat').max().alias('last_date'))
    .with_columns((pl.lit(SUB_CUTOFF) - pl.col('last_date')).dt.total_days().alias('days_since_last_purchase_article'))
    .drop('last_date')
)

customers_age_pl = customers.select(['customer_id', 'age_bin'])
df_age_pl = df_train_pl.join(customers_age_pl, on='customer_id', how='left')
age_bin_counts_pl = df_age_pl.group_by(['age_bin', 'article_id']).agg(pl.len().alias('cnt'))
age_bin_totals_pl = df_age_pl.group_by('age_bin').agg(pl.len().alias('total'))
age_bin_rate_pl = (
    age_bin_counts_pl.join(age_bin_totals_pl, on='age_bin', how='left')
    .with_columns((pl.col('cnt') / pl.col('total')).alias('age_bin_purchase_rate'))
    .select(['age_bin', 'article_id', 'age_bin_purchase_rate'])
)

art_cats_pl = articles.select(['article_id', 'product_type_no', 'garment_group_no', 'index_group_no'])
df_cat_pl = df_train_pl.join(art_cats_pl, on='article_id', how='left')
user_total_pl = df_train_pl.group_by('customer_id').agg(pl.len().alias('user_total'))

user_cat_pls = {}
for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
    uc = (
        df_cat_pl.group_by(['customer_id', cat_col]).agg(pl.len().alias(f'user_{cat_col}_count'))
        .join(user_total_pl, on='customer_id', how='left')
        .with_columns((pl.col(f'user_{cat_col}_count') / pl.col('user_total')).alias(f'user_{cat_col}_ratio'))
        .drop('user_total')
    )
    user_cat_pls[cat_col] = uc

last_cat_pls = {}
for cat_col in ['product_type_no', 'garment_group_no']:
    lc = (
        df_cat_pl.group_by(['customer_id', cat_col])
        .agg(pl.col('t_dat').max().alias('last_cat_date'))
        .with_columns((pl.lit(SUB_CUTOFF) - pl.col('last_cat_date')).dt.total_days().alias(f'days_since_last_{cat_col}'))
        .drop('last_cat_date')
    )
    last_cat_pls[cat_col] = lc

source_names = ['timedecay', 'repurchase', 'variant', 'also_bought',
                'item2item_cf', 'bin_popular', 'purchase_interval']
source_pls = {}
for name in source_names:
    path = CANDIDATE_DIR / f"{name}_{SUB_CUTOFF}.parquet"
    if path.exists():
        source_pls[name] = (
            pl.read_parquet(path, columns=['customer_id', 'article_id'])
            .unique()
            .with_columns(pl.lit(1).alias(f'is_{name}'))
        )
    else:
        source_pls[name] = None

del df_train_pl, df_age_pl, df_cat_pl; gc.collect()
print("  事前計算完了!")

# バッチ処理
BATCH_SIZE = 100000
all_customer_ids = scored_df['customer_id'].unique().to_list()
all_predictions = {}

for batch_start in range(0, len(all_customer_ids), BATCH_SIZE):
    batch_cids = all_customer_ids[batch_start:batch_start+BATCH_SIZE]
    batch_end = min(batch_start + BATCH_SIZE, len(all_customer_ids))
    batch_num = batch_start // BATCH_SIZE + 1
    print(f"\n  バッチ {batch_num}: {batch_start:,}〜{batch_end:,}")
    
    batch = scored_df.filter(pl.col('customer_id').is_in(batch_cids))
    
    for name in source_names:
        if source_pls[name] is not None:
            batch = batch.join(source_pls[name], on=['customer_id', 'article_id'], how='left').with_columns(
                pl.col(f'is_{name}').fill_null(0)
            )
        else:
            batch = batch.with_columns(pl.lit(0).alias(f'is_{name}'))
    
    batch = batch.join(repeat_count_pl, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('repeat_count').fill_null(0)
    )
    batch = batch.join(last_purchase_article_pl, on=['customer_id', 'article_id'], how='left').with_columns(
        pl.col('days_since_last_purchase_article').fill_null(-1)
    )
    batch = batch.join(customers_age_pl, on='customer_id', how='left')
    batch = batch.join(age_bin_rate_pl, on=['age_bin', 'article_id'], how='left').with_columns(
        pl.col('age_bin_purchase_rate').fill_null(0)
    )
    batch = batch.drop('age_bin')
    
    batch = batch.join(art_cats_pl, on='article_id', how='left')
    for cat_col in ['product_type_no', 'garment_group_no', 'index_group_no']:
        batch = batch.join(user_cat_pls[cat_col], on=['customer_id', cat_col], how='left').with_columns([
            pl.col(f'user_{cat_col}_count').fill_null(0),
            pl.col(f'user_{cat_col}_ratio').fill_null(0),
        ])
    for cat_col in ['product_type_no', 'garment_group_no']:
        batch = batch.join(last_cat_pls[cat_col], on=['customer_id', cat_col], how='left').with_columns(
            pl.col(f'days_since_last_{cat_col}').fill_null(-1)
        )
    batch = batch.drop(['product_type_no', 'garment_group_no', 'index_group_no'])
    
    batch_pd = batch.to_pandas()
    del batch; gc.collect()
    
    batch_pd = batch_pd.merge(user_feat_pd, on='customer_id', how='left')
    batch_pd = batch_pd.merge(article_feat_pd, on='article_id', how='left')
    
    for col in feature_cols:
        if col in batch_pd.columns:
            batch_pd[col] = batch_pd[col].fillna(0)
    
    batch_pd['pred_score'] = model_sub.predict(batch_pd[feature_cols])
    
    top12 = (
        batch_pd
        .sort_values(['customer_id', 'pred_score'], ascending=[True, False])
        .groupby('customer_id')
        .head(12)
    )
    for cid, group in top12.groupby('customer_id'):
        arts = group['article_id'].tolist()
        all_predictions[cid] = ' '.join([article_id_reverse[a] for a in arts])
    
    del batch_pd, top12; gc.collect()
    print(f"    完了: {len(batch_cids):,}ユーザー（累計: {len(all_predictions):,}）")

# 全ユーザーにマージ
print(f"\n予測済みユーザー数: {len(all_predictions):,}")

bin_pop = pl.read_parquet(str(CANDIDATE_DIR / f'bin_popular_{SUB_CUTOFF}.parquet'))
fallback_arts = bin_pop.sort('rank').select('article_id').head(12).to_series().to_list()
fallback_str = ' '.join([article_id_reverse[a] for a in fallback_arts])

rows = []
for cid_str, cid_int in zip(customer_id_map.keys(), customer_id_map.values()):
    pred = all_predictions.get(cid_int, fallback_str)
    rows.append({'customer_id': cid_str, 'prediction': pred})

submission = pd.DataFrame(rows)
submission.to_csv('/workspace/outputs/submissions/submission_rerank_v7.csv', index=False)
print(f"\n保存完了: submission_rerank_v6.csv")
print(f"ユーザー数: {len(submission):,}")
print(submission.head(5))
print("\n提出ファイル作成完了!")


提出ファイル作成中...
  提出用cutoff: 2020-09-22
  全候補数: 122,600,347行
  ユーザー特徴量: キャッシュ読み込み /home/rapids/features/user_features_2020-09-22.parquet
  商品特徴量: キャッシュ読み込み /home/rapids/features/article_features_2020-09-22.parquet
  事前計算中...
  事前計算完了!

  バッチ 1: 0〜100,000
    完了: 100,000ユーザー（累計: 100,000）

  バッチ 2: 100,000〜200,000
    完了: 100,000ユーザー（累計: 200,000）

  バッチ 3: 200,000〜300,000
    完了: 100,000ユーザー（累計: 300,000）

  バッチ 4: 300,000〜400,000
    完了: 100,000ユーザー（累計: 400,000）

  バッチ 5: 400,000〜500,000
    完了: 100,000ユーザー（累計: 500,000）

  バッチ 6: 500,000〜600,000
    完了: 100,000ユーザー（累計: 600,000）

  バッチ 7: 600,000〜700,000
    完了: 100,000ユーザー（累計: 700,000）

  バッチ 8: 700,000〜800,000
    完了: 100,000ユーザー（累計: 800,000）

  バッチ 9: 800,000〜900,000
    完了: 100,000ユーザー（累計: 900,000）

  バッチ 10: 900,000〜1,000,000
    完了: 100,000ユーザー（累計: 1,000,000）

  バッチ 11: 1,000,000〜1,100,000
    完了: 100,000ユーザー（累計: 1,100,000）

  バッチ 12: 1,100,000〜1,200,000
    完了: 100,000ユーザー（累計: 1,200,000）

  バッチ 13: 1,200,000〜1,300,000
    完了: 100,000